## Multi-tenant row-level security setup

Sets up RLS by drug therapy area + PII column mask on patient_age.

**Prerequisites** — these UC account groups must exist (create in account console or via SCIM):
- `admins` — sees all data, raw PII
- `clinical_full_access` — sees all data, raw PII (for clinical reviewers)
- `tenant_cardio`, `tenant_metabolic`, `tenant_psych` — see only their drug therapy area, masked PII

Run as a workspace admin. **Idempotent** — safe to re-run.

In [0]:
# 1. Create tenant → drug mapping table
spark.sql("""
CREATE TABLE IF NOT EXISTS fda_rag.gold._tenant_drug_map (
    tenant_group STRING NOT NULL,
    drug_generic STRING NOT NULL,
    therapy_area STRING
) USING DELTA
TBLPROPERTIES (
    'delta.minReaderVersion' = '3',
    'delta.minWriterVersion' = '7'
)
""")

spark.sql("DELETE FROM fda_rag.gold._tenant_drug_map")
spark.sql("""
INSERT INTO fda_rag.gold._tenant_drug_map VALUES
    ('tenant_cardio',     'warfarin',     'cardiovascular'),
    ('tenant_cardio',     'atorvastatin', 'cardiovascular'),
    ('tenant_cardio',     'lisinopril',   'cardiovascular'),
    ('tenant_cardio',     'amlodipine',   'cardiovascular'),
    ('tenant_metabolic',  'metformin',    'metabolic'),
    ('tenant_metabolic',  'atorvastatin', 'metabolic'),
    ('tenant_psych',      'sertraline',   'psychiatric'),
    ('tenant_otc',        'ibuprofen',    'over-the-counter'),
    ('tenant_otc',        'omeprazole',   'over-the-counter')
""")

# Restrict who can read the mapping table itself (only admins)
spark.sql("REVOKE ALL PRIVILEGES ON TABLE fda_rag.gold._tenant_drug_map FROM `account users`")
spark.sql("GRANT SELECT ON TABLE fda_rag.gold._tenant_drug_map TO `admins`")

display(spark.sql("SELECT * FROM fda_rag.gold._tenant_drug_map ORDER BY therapy_area, drug_generic"))

In [0]:
# 2. Row filter function — checks group membership against mapping table
spark.sql("""
CREATE OR REPLACE FUNCTION fda_rag.gold.tenant_drug_filter(drug_in STRING)
RETURNS BOOLEAN
RETURN
  is_account_group_member('admins')
  OR EXISTS (
    SELECT 1
    FROM fda_rag.gold._tenant_drug_map m
    WHERE is_account_group_member(m.tenant_group)
      AND lower(m.drug_generic) = lower(drug_in)
  )
""")

# 3. Column mask function for patient_age PII
spark.sql("""
CREATE OR REPLACE FUNCTION fda_rag.gold.age_pii_mask(age INT)
RETURNS INT
RETURN CASE
  WHEN is_account_group_member('admins') THEN age
  WHEN is_account_group_member('clinical_full_access') THEN age
  WHEN age IS NULL THEN NULL
  ELSE CAST(FLOOR(age / 10) * 10 AS INT)
END
""")

print("Functions created:")
display(spark.sql("SHOW FUNCTIONS IN fda_rag.gold LIKE '*tenant*'"))
display(spark.sql("SHOW FUNCTIONS IN fda_rag.gold LIKE '*pii*'"))

In [0]:
# 4. Apply row filter to all tables that have drug_generic
tables_with_drug = [
    ("fda_rag.silver.drug_labels",    "drug_generic"),
    ("fda_rag.silver.adverse_events", "drug_name"),
    ("fda_rag.gold.fda_chunks",       "drug_generic"),
    ("fda_rag.gold.dim_drug",         "drug_generic"),
]

for table, column in tables_with_drug:
    spark.sql(f"""
        ALTER TABLE {table}
        SET ROW FILTER fda_rag.gold.tenant_drug_filter ON ({column})
    """)
    print(f"  Row filter applied: {table} on {column}")

# 5. Apply column mask to PII column
spark.sql("""
    ALTER TABLE fda_rag.silver.adverse_events
    ALTER COLUMN patient_age
    SET MASK fda_rag.gold.age_pii_mask
""")
print("  Column mask applied: silver.adverse_events.patient_age")

# 6. fact_adverse_event uses surrogate key (drug_key), not drug_generic directly.
# For full tenant isolation on the fact, denormalize drug_generic during fact build,
# then apply the same filter. For this demo, dim_drug is filtered which prevents
# joins from returning unauthorized drug names.
print("\nNote: fact_adverse_event is filtered transitively via dim_drug joins.")
print("For full isolation, add drug_generic column to fact and apply filter directly.")

In [0]:
# 7. Verification queries — run these AS DIFFERENT USERS to confirm isolation

# As admin: should see all drugs
print("As admin (you):")
display(spark.sql("""
    SELECT drug_generic, COUNT(*) AS n_records
    FROM fda_rag.silver.drug_labels
    GROUP BY drug_generic
    ORDER BY drug_generic
"""))

# Show patient_age behavior — admin sees raw values
print("\nPII mask check (admin sees raw ages):")
display(spark.sql("""
    SELECT patient_age, COUNT(*) AS n
    FROM fda_rag.silver.adverse_events
    WHERE patient_age IS NOT NULL
    GROUP BY patient_age
    ORDER BY patient_age
    LIMIT 10
"""))

print("\nTo test as a tenant user:")
print("  1. Add a test user to a tenant_* group via account console")
print("  2. Have them run: SELECT DISTINCT drug_generic FROM fda_rag.silver.drug_labels")
print("  3. They should see only drugs mapped to their group")

## RAG chain update for tenant-filtered retrieval

**Vector indexes don't inherit row filters.** Apply the tenant filter at query time. Update notebook `04_rag_chain` retriever to:

```python
from databricks.sdk.runtime import dbutils
import os

def _get_user_drug_filter():
    user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
    rows = spark.sql(f\"\"\"
        SELECT DISTINCT drug_generic
        FROM fda_rag.gold._tenant_drug_map m
        WHERE is_account_group_member(m.tenant_group)
    \"\"\").collect()
    drugs = [r.drug_generic for r in rows]
    return f\"drug_generic IN ({','.join(repr(d) for d in drugs)})\" if drugs else \"1=0\"

retriever = DatabricksVectorSearch(
    endpoint=\"fda-vs-endpoint\",
    index_name=\"fda_rag.gold.fda_chunks_index\",
    text_column=\"text\",
    columns=[\"drug_generic\", \"drug_brand\", \"section\"],
).as_retriever(search_kwargs={
    \"k\": 5,
    \"filter\": _get_user_drug_filter()
})
```

When deployed behind Model Serving, the endpoint executes as a service principal — pass the calling user through `auth_context` and look up *that* user's allowed drugs instead.